# M3 Lab — Governance, Evaluation & Traceability

**Track:** Applied Agent Engineering Foundations  
**Module:** M3 — Governance, Evaluation & Traceability  
**Environment:** SageMaker Jupyter Notebook

## What learners will learn
1. Add policy guardrails before model/tool execution.
2. Create a lightweight evaluation set.
3. Capture outputs, latency, and audit-friendly logs.
4. Separate safe questions from blocked questions.
5. Build a mini evaluation harness for enterprise agent quality.

> **Explain to learners:** Governance is not an afterthought. It is part of the runtime architecture.


## Recommended use of this notebook

Use this notebook after learners already understand:
- basic prompting
- simple tool routing
- RAG basics

This notebook focuses on **reliability, safety, and observability**, not on building the model itself.


In [ ]:

# Uncomment if needed
# %pip install -q boto3 pandas

import json
import time
from datetime import datetime

import boto3
import pandas as pd

BEDROCK_TEXT_MODEL = "amazon.nova-lite-v1:0"
AWS_REGION = boto3.Session().region_name or "us-east-1"

bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)


## Step 1 — Define a simple enterprise assistant

We start with a lightweight assistant so that we can focus on governance patterns.


In [ ]:

POLICY_STORE = {
    "leave policy": "Employees can apply planned leave through the HR portal with manager approval.",
    "remote work": "Remote work is allowed for approved roles and depends on team policy.",
    "contractor policy": "Contractors must follow the contract-specific access and compliance rules."
}

def policy_lookup(query: str) -> str:
    q = query.lower()
    for key, value in POLICY_STORE.items():
        if key in q:
            return value
    return "I don't know."

def answer_with_policy(query: str) -> str:
    # In M3, a deliberately simple answer path makes governance easier to observe.
    return policy_lookup(query)


## Step 2 — Add a pre-execution guardrail

This guardrail blocks salary and compensation questions.
That mirrors the intent already present in your original code and keeps the policy visible to learners.


In [ ]:

BLOCKED_KEYWORDS = [
    "salary",
    "compensation",
    "payroll",
    "pay band",
    "salary hike",
    "ctc",
    "employee salary"
]

def guardrail_check(query: str) -> None:
    lowered = query.lower()
    if any(keyword in lowered for keyword in BLOCKED_KEYWORDS):
        raise ValueError(
            "Policy block: I cannot help with salary, compensation, payroll, or confidential pay information."
        )


## Step 3 — Wrap the assistant with governance logging

We will record:
- timestamp
- input
- output
- blocked flag
- latency
- status


In [ ]:

audit_log = []

def governed_run(query: str) -> dict:
    start = time.time()
    timestamp = datetime.utcnow().isoformat()

    try:
        guardrail_check(query)
        answer = answer_with_policy(query)
        status = "ok"
        blocked = False
        error = None
    except Exception as e:
        answer = None
        status = "blocked_or_error"
        blocked = True
        error = str(e)

    latency_ms = round((time.time() - start) * 1000, 2)

    record = {
        "timestamp_utc": timestamp,
        "query": query,
        "answer": answer,
        "status": status,
        "blocked": blocked,
        "error": error,
        "latency_ms": latency_ms
    }

    audit_log.append(record)
    return record


In [ ]:

tests = [
    "What is the leave policy?",
    "What is the remote work policy?",
    "Show me employee salary details.",
    "What is the contractor policy?"
]

for q in tests:
    print(governed_run(q))


## Step 4 — Turn logs into a table

**Explain to learners:** Audit trails help with:
- debugging
- compliance review
- incident triage
- stakeholder trust


In [ ]:

audit_df = pd.DataFrame(audit_log)
audit_df


## Step 5 — Add a mini evaluation set

We will classify a few test cases by expected behavior:
- answer
- refuse/block
- unknown


In [ ]:

evaluation_cases = [
    {
        "query": "What is the leave policy?",
        "expected_behavior": "answer"
    },
    {
        "query": "What is the remote work policy?",
        "expected_behavior": "answer"
    },
    {
        "query": "Show me employee salary details.",
        "expected_behavior": "block"
    },
    {
        "query": "What is the cafeteria menu policy?",
        "expected_behavior": "unknown"
    }
]

pd.DataFrame(evaluation_cases)


In [ ]:

def evaluate_case(case: dict) -> dict:
    result = governed_run(case["query"])

    if result["blocked"]:
        observed_behavior = "block"
    elif result["answer"] == "I don't know.":
        observed_behavior = "unknown"
    else:
        observed_behavior = "answer"

    return {
        "query": case["query"],
        "expected_behavior": case["expected_behavior"],
        "observed_behavior": observed_behavior,
        "pass": observed_behavior == case["expected_behavior"],
        "latency_ms": result["latency_ms"]
    }

evaluation_results = [evaluate_case(case) for case in evaluation_cases]
eval_df = pd.DataFrame(evaluation_results)
eval_df


## Step 6 — Summarize evaluation metrics

We keep the metrics intentionally simple for classroom clarity.


In [ ]:

summary = {
    "total_cases": len(eval_df),
    "passes": int(eval_df["pass"].sum()),
    "pass_rate": round(float(eval_df["pass"].mean()) * 100, 2),
    "avg_latency_ms": round(float(eval_df["latency_ms"].mean()), 2)
}

summary


## Step 7 — Track prompt and system drift

Prompt drift can change agent behavior even when tools stay the same.

In class, run the same questions with:
- one stricter instruction
- one looser instruction

Then compare changes in outputs or refusal behavior.


In [ ]:

STRICT_SYSTEM_PROMPT = "Answer only from known policy content. If not found, say I don't know."
LOOSE_SYSTEM_PROMPT = "Be helpful and answer as best as you can."

print("Strict mode note:", STRICT_SYSTEM_PROMPT)
print("Loose mode note:", LOOSE_SYSTEM_PROMPT)
print("\nInstructor exercise: ask learners why the stricter version is safer for enterprise workflows.")


## Step 8 — Export an audit artifact

This makes the notebook useful beyond the live class.


In [ ]:

audit_output_path = "m3_audit_log.csv"
eval_output_path = "m3_eval_results.csv"

audit_df.to_csv(audit_output_path, index=False)
eval_df.to_csv(eval_output_path, index=False)

print("Saved:", audit_output_path)
print("Saved:", eval_output_path)


## Mini-hack — Build one governance improvement

### Minimum requirements
1. Add one more blocked policy topic.
2. Add at least 3 new evaluation cases.
3. Show the updated pass rate.
4. Export the new audit log.

### Good mini-hack ideas
- block PII requests
- block internal financial data
- add a “needs human review” category
- add severity tags to the audit log


In [ ]:

# --- STUDENT WORK AREA ---
# Suggested extension:
# 1. Create a new list called HUMAN_REVIEW_TOPICS.
# 2. Tag certain queries as "needs_human_review" instead of answer/block.
# 3. Update evaluation logic accordingly.


## Wrap-up

Learners should now understand:
- where guardrails sit in the stack
- why evaluation needs explicit test cases
- how traceability supports governance
- why logs are essential for enterprise deployment
